<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>Simulation-Based Inference</h1>

<h2>Structure</h2>

| Part | Topic | Type |
|:----:|-------|------|
| 0 | Recap | Conceptual |
| 1 | Worked example: projectile inference | Guided |
| 2 | Exercises: pick your track | Hands-on |
| 3 | Posterior predictive checks | Conceptual |
| 4 | Simulation-based calibration | Conceptual + Exercise |

<h2>Preamble</h2>

<ul>
<li>You will be required to write code. Skeleton functions with <code># TODO</code> comments are provided, along with hint and solution blocks.</li>
<li>This notebook assumes you have read and run the pre-tutorial notebook. If you haven't, the recap in §0 will not be sufficient.</li>
<li>Imports: run the cell below before starting to ensure you have everything installed.</li>
</ul>

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from tqdm.notebook import tqdm

from sbi.inference import NPE
from sbi.utils import BoxUniform
from sbi.analysis import pairplot
from sbi.diagnostics import run_sbc
from sbi.analysis.plot import sbc_rank_plot

import corner
import zuko

print('All imports done.')
torch.manual_seed(42)
np.random.seed(42)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§0. Recap</h1>

Many models in astrophysics are implicit. This means that you can run the model/simulation forward ($\theta \to \mathbf{x}$), but the likelihood $p(\mathbf{x} \mid \theta)$ either requires marginalising over a high-dimensional latent space, and/or cannot be written down or evaluated, and/or the likelihood is very computationally demanding.

Standard MCMC and nested sampling require a tractable likelihood, so they cannot be used directly, without simplifying the likelihood extensively.

An older way around this was Approximate Bayesian Computation (ABC). For this, you would create some sort of summary statistic $s(\mathbf{x})$, and then draw $\theta$ from the prior, simulate/run the forward model and keep $\theta$ if the two summary statistics were 'close': $|s(\mathbf{x}) - s(\mathbf{x}')| < \varepsilon$. 

This method works in principle, but has two main issues:

1. If you want your posterior to be accurate, $\varepsilon$ must be small. However, this means that the chance of acceptance is very small, meaning you need to run the forwards simulation many times.
2. Most summary statistics lose information, meaning that the posterior is only approximately the true posterior. These summary statistics must also be hand-crafted, which can be difficult.

*Neural SBI* solves these issues by replacing the summary statistic creation and the accept/reject step with a neural network trained on $(\theta, \mathbf{x})$ pairs drawn from the prior and simulator.

Three main types exist:
1. Neural Posterior Estimator (NPE): learns the posterior $q_\phi(\theta\mid\mathbf{x})$ directl
2. Neural Likelihood Estimator (NLE): learns the likelihood $q_\phi(\mathbf{x}\mid\theta)$. This still needs some sort of sampling (e.g. MCMC) to learn the posterior
3. Neural Ratio Estimator (NRE): learns the likelihood-to-evidence ratio via classification (which for some reason, isn't technically classification)

*Note: here we have written $q_\phi()$ rather than $p()$. The $q$ represents that this is an approximation of $p$, and the $_\phi$ indicates that the density is constructed from the weights of the neural net.*

We will focus on NPEs, as they are the coolest in my opinion.

------------
<h2>§0.1 The Neural Network / Density Estimator</h2>

In SBI, the neural network is called the density estimator (because it estimates the density). The network must output a full density distribution and not just a point estimate. In the before tutorial, we looked at two versions of density estimators:

1. Mixture Density Network (MDNs): The network outputs the relative weights, means, and covariances of a Gaussian mixture, which creates the density. This is simple, but is limited to elliptical blobs with Gaussian tails.
2. Normalising Flows: an invertible map from a base Gaussian to an arbitrarily complex target, using the change-of-variables formula to evaluate the density exactly.  There are a few conditions behind the scences to make this computationally tractable, with the main one being to ensure that the jacobian of the transformation is easy to compute (by making the transfomration matrix triangular).

Typically some sort of flow is used.

<h2>§0.2 Conditional Flows</h2>

To turn a flow into something useful for inference, we need to make it conditional (on the data). The two-moons flow from the pre-tutorial (and below) learns an unconditional density $q_\phi(\theta)$ It warps a Gaussian into the same fixed target shape every time. But for SBI, the posterior will change depending on what data you observed (higher SNR means tighter posteriors). We need a flow that outputs a different density for each $\mathbf{x}$. To solve this, you make the warping function depend on $\mathbf{x}$.

Recall that a flow maps a sample $z$ frawn from a simple base distribution (e.g. a Gaussian) through an invertible transformation to produce $\theta = f_\phi(z)$. Each layer of the flow has internal neural netowrks that produce the scale and shift parameters for that layers transformation. In an unconditional flow, the networks only take $z$ as an input. In a conditional flow, they also take $x$ as an input. So the flow becomes $\theta = f_\phi(z, \mathbf{x})$.

<h2>§0.3 Embedding Networks</h2>

When $\mathbf{x}$ is high-dimensional (a spectrum or an image, for example), you don't feed it directly into the flow network. Instead you use an embedding network $h_\varphi(\mathbf{x})$ which first compresses it to a low-dimensional summary vector, and condition the flow network on that. This is akin to creating the summary statistic in SBI, except here the summary statistic is learnt. Note that the embedding network is trained end-to-end with the flow.

Below is a pictorial explanation of the SBI process.

<img src="SBI_Step1.png" style="max-width: 100%;">
<img src="SBI_Step2.png" style="max-width: 100%;">

Once a flow is trained, inference on a new observation is a single forwards pass (milliseconds)

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§1 Worked Example: Projectile Inference</h1>

Let's walk through a complete SBI pipeline on : projectile motion.

<h2>The problem</h2>

A projectile is launched from the origin with unknown speed $v_0$ and launch angle $\alpha$. It follows the standard equations of motion (no drag) until it hits the ground:

$$x(t) = v_0 \cos\alpha \; t, \qquad y(t) = v_0 \sin\alpha \; t - \tfrac{1}{2}g\,t^2$$

We observe the trajectory as a set of noisy $(x, y)$ positions. Given a single observed trajectory, we want to infer $v_0$ and $\alpha$.

This is a problem where you could write down the likelihood (it's a Gaussian in the residuals), so you could use MCMC. We're using it as a worked example because we can check the SBI posterior against the truth. In practice, you would use SBI when the simulator is too complex for a tractable likelihood.

<h2>The simulator</h2>

The simulator takes $\theta = (v_0, \alpha)$, computes the full trajectory, finds the impact time (when $y = 0$), and returns $n_{\rm obs}$ equally-spaced snapshots between launch and impact with added Gaussian noise. We want the data to have the same length as it is easier to train a network this way. Because we always return the same number of snapshots, the data vector $\mathbf{x}$ is always $2 \times n_{\rm obs}$-dimensional, regardless of the flight time. The flight duration is implicitly encoded in the spacing of the points.

</div>

In [ ]:
####################
# Create simulator #
####################

g = 9.81          # m/s²
n_obs = 20        # number of equally-spaced snapshots per trajectory
noise_std = 1   # measurement noise


def simulate_projectile(theta):
    """
    Simulate a noisy projectile trajectory.
    
    Parameters
    ----------
    theta : torch.Tensor, shape (2,)
        theta[0] = v0     (launch speed, m/s)
        theta[1] = alpha   (launch angle, radians)
    
    Returns
    -------
    x_obs : torch.Tensor, shape (2 * n_obs,)
        Flattened (x, y) positions at n_obs equally-spaced times from launch to impact.
    """
    v0    = theta[0].item()
    alpha = theta[1].item()
    
    # Time of flight (when y = 0 again)
    t_flight = 2 * v0 * np.sin(alpha) / g
    
    # Guard against zero or negative flight time
    if t_flight <= 0:
        return torch.zeros(2 * n_obs)
    
    # Equally-spaced time samples from 0 to t_flight
    t = np.linspace(0, t_flight, n_obs)
    
    # True positions
    x = v0 * np.cos(alpha) * t
    y = v0 * np.sin(alpha) * t - 0.5 * g * t**2
    
    # Add noise
    x_noisy = x + noise_std * np.random.randn(n_obs)
    y_noisy = y + noise_std * np.random.randn(n_obs)
    
    # Flatten into a single data vector
    data = np.concatenate([x_noisy, y_noisy])
    return torch.tensor(data, dtype=torch.float32)


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

Let's generate a single "observed" trajectory with known ground truth, and plot it.

</div>

In [ ]:
####################################
# §0.5.2 Generate a fake observation #
####################################
torch.manual_seed(0)
np.random.seed(0)

# Ground truth
v0_true    = 25.0               # m/s
alpha_true = np.radians(55.0)   # 55 degrees

theta_true = torch.tensor([v0_true, alpha_true])
x_obs = simulate_projectile(theta_true)

# Unpack for plotting
x_pos = x_obs[:n_obs].numpy()
y_pos = x_obs[n_obs:].numpy()

# Also compute the noiseless trajectory for reference
t_flight_true = 2 * v0_true * np.sin(alpha_true) / g
t_true = np.linspace(0, t_flight_true, 300)
x_true = v0_true * np.cos(alpha_true) * t_true
y_true = v0_true * np.sin(alpha_true) * t_true - 0.5 * g * t_true**2

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_true, y_true, 'k--', lw=1, alpha=0.4, label='True trajectory')
ax.scatter(x_pos, y_pos, c='C0', s=30, zorder=5, label='Noisy observations')
ax.axhline(0, color='0.5', lw=0.5)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title(rf'Observed trajectory  ($v_0 = {v0_true}$ m/s, $\alpha = 55°$)')
ax.legend()
ax.set_ylim(bottom=-2)
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">


Now we set up the SBI pipeline. The three ingredients are:

<ol>
<li>A <b>prior</b> over the parameters. We use a uniform prior: $v_0 \in [10, 45]$ m/s and $\alpha \in [0.2, 1.4]$ rad ($\approx$ 11° to 80°).</li>
<li>The <b>simulator</b> we just wrote.</li>
<li>An <b>inference object</b> (here, NPE), which handles the flow architecture and training loop internally.</li>
</ol>

</div>

In [ ]:
# 1. Prior
prior = BoxUniform(
    low=torch.tensor([10.0, 0.2]),
    high=torch.tensor([45.0, 1.4])
)

# 2. Generate training simulations
n_sims = 10_000
thetas = prior.sample((n_sims,))
# add a whole bunch of simulated data to a torch array
xs = torch.stack([simulate_projectile(t) for t in tqdm(thetas, desc='Simulating')])

# 3. Train NPE
inference = NPE(prior=prior)  # define object, give prior
inference.append_simulations(thetas, xs) # add simulations
density_estimator = inference.train(show_train_summary=True) # train it

# 4. Build the posterior object
posterior = inference.build_posterior(density_estimator)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

Now we condition on our observed trajectory and draw posterior samples. This is a single forward pass through the trained flow, and takes milliseconds.

</div>

In [ ]:
samples = posterior.sample((10_000,), x=x_obs) 

# Convert alpha to degrees for interpretability
samples_plot = samples.numpy().copy()
samples_plot[:, 1] = np.degrees(samples_plot[:, 1])

fig = corner.corner(
    samples_plot,
    labels=[r'$v_0$ (m/s)', r'$\alpha$ (degrees)'],
    truths=[v0_true, 55.0],
    truth_color='C3',
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt='.1f',
)
plt.suptitle('NPE Posterior', y=1.02, fontsize=14)
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">


A corner plot shows us the marginals, but we should also check: do trajectories drawn from the posterior actually <i>look</i> like our data? This is a posterior predictive check: draw parameters from the posterior, simulate trajectories, and overlay them on the observation. If the posterior is good, the observed data should sit comfortably within the cloud of simulated trajectories.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

# Draw 200 posterior samples and simulate noiseless trajectories
n_pp = 200
pp_samples = posterior.sample((n_pp,), x=x_obs).numpy()

for i in range(n_pp):
    v0_i    = pp_samples[i, 0]
    alpha_i = pp_samples[i, 1]
    t_f = 2 * v0_i * np.sin(alpha_i) / g
    if t_f <= 0:
        continue
    t_i = np.linspace(0, t_f, 200)
    x_i = v0_i * np.cos(alpha_i) * t_i
    y_i = v0_i * np.sin(alpha_i) * t_i - 0.5 * g * t_i**2
    ax.plot(x_i, y_i, color='C0', alpha=0.03, lw=1)

# Overlay observations
ax.scatter(x_pos, y_pos, c='C3', s=30, zorder=5, label='Observed data', edgecolors='k', linewidths=0.5)
ax.plot(x_true, y_true, 'k--', lw=1.5, alpha=0.5, label='True trajectory')
ax.axhline(0, color='0.5', lw=0.5)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Posterior predictive check: 200 trajectories from the posterior')
ax.legend()
ax.set_ylim(bottom=-2)
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h4>What just happened</h4>

<ol>
<li>We wrote a simulator: a function that takes $(v_0, \alpha)$ and returns a noisy trajectory. This is the only model specification SBI needs.</li>
<li>We defined a prior: a box uniform over physically reasonable parameter ranges.</li>
<li>We generated 10,000 training pairs $(\theta_i, \mathbf{x}_i)$ by sampling from the prior and running the simulator.</li>
<li>The <code>sbi</code> package trained a conditional normalising flow (the NPE) to learn $q_\phi(\theta \mid \mathbf{x})$ from these pairs.</li>
<li>We conditioned on our observation and obtained 10,000 posterior samples in milliseconds. No MCMC, no likelihood evaluation.</li>
<li>The posterior predictive check confirmed that the inferred posterior produces trajectories consistent with the data.</li>
</ol>

Note the degeneracy visible in the corner plot: a slightly faster launch at a slightly lower angle can produce a similar trajectory to a slower launch at a higher angle. The posterior captures this correlation. This is something SBI does naturally, since the flow learns the full joint posterior.

Also note: this posterior is amortised<. If you observe a second trajectory tomorrow, you just call <code>posterior.sample(..., x=x_obs_new)</code> and you get a new posterior in milliseconds, without retraining. This is the advantage of NPE over methods like MCMC, where every new observation requires a new chain.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§2 Exercises </h1>

I have written two exercises that cover the same idea. The first track is a exoplanet transit model, and the second is SED fitting. Choose whichever track is more interesting to you. Both follow the same structure as the worked projectile example:


<ol>
<li>Write a simulator</li>
<li>Define a prior</li>
<li>Generate training simulations</li>
<li>Train an NPE</li>
<li>Sample the posterior for a given observation</li>
<li>Make a corner plot</li>
<li>Perform a posterior predictive check</li>
</ol>
If you finish one, try the other.

<table style="width: 100%; border-collapse: collapse; margin: 20px 0;">
<tr>
<th style="padding: 10px; text-align: center; border: 1px solid rgba(0,64,133,0.3);">Track A: Stellar SED Fitting</th>
<th style="padding: 10px; text-align: center; border: 1px solid rgba(0,64,133,0.3);">Track B: Exoplanet Transit</th>
</tr>
<tr>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);">Simulate broadband <i>ugriz</i> photometry from a blackbody + dust model, infer $T_{\text{eff}}$, $\log_{10}\Omega$, $A_V$. Three parameters, $T$-$A_V$ degeneracy.</td>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);">Infer the radius ratio and impact parameter of a transiting exoplanet from a noisy light curve. Two parameters, banana-shaped degeneracy.</td>
</tr>
</table>

</div>


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>Track A: Stellar SED Fitting</h1>

<h2>The problem</h2>

You observe a star's broadband photometry in five filters (<i>ugriz</i>). The intrinsic spectrum is a Planck function scaled by the solid angle $\Omega = (R/d)^2$, but the observed flux is reddened by interstellar dust (I know this isn't realistic). The three parameters to infer are:

<ul>
<li>$T_\text{eff}$: the effective temperature, which sets the shape of the blackbody.</li>
<li>$\log_{10}\Omega$: the log solid angle, which sets the overall flux level.</li>
<li>$A_V$: the V-band extinction, which reddens and dims the spectrum.</li>
</ul>

The interesting degeneracy here is between $T_\text{eff}$ and $A_V$: a cooler star looks similar to a hotter star behind more dust, because both make the SED redder. With only broadband photometry this degeneracy can be significant.

We use the Cardelli, Clayton &amp; Mathis (1989)* extinction law with $R_V = 3.1$, evaluated at the effective wavelengths of each filter. This is a gross simplification of real SED fitting (no spectral lines, no model atmospheres), but the SBI workflow is the same as what you would do with a full stellar model grid.

\*Note: This is just what Claude says, I have no idea if this is true, someone correct me

<h2>Step 1: Write the simulator</h2>

The extinction law and filter wavelengths are provided. Your job is to write the simulator function that takes $\theta = (T_\text{eff}, \log_{10}\Omega, A_V)$ and returns five noisy broadband fluxes.

</div>

In [ ]:
# Physical constants (SI)
h_pl = 6.626e-34    # Planck constant
c_l  = 3.0e8        # speed of light
kB   = 1.381e-23    # Boltzmann constant

# SDSS ugriz effective wavelengths (metres)
filter_names = ['u', 'g', 'r', 'i', 'z']
filter_wavelengths = np.array([354.3, 477.0, 623.1, 762.5, 913.4]) * 1e-9

In [ ]:
def cardelli_extinction(wavelength_m, R_V=3.1):
    """Compute A_lambda / A_V using Cardelli+89 for optical/NIR."""
    # Convert to inverse microns
    x = 1.0 / (wavelength_m * 1e6)
    
    # Optical/NIR range: 1.1 <= x <= 3.3 inverse microns
    y = x - 1.82
    a = (1 + 0.17699*y - 0.50447*y**2 - 0.02427*y**3 + 0.72085*y**4
         + 0.01979*y**5 - 0.77530*y**6 + 0.32999*y**7)
    b = (1.41338*y + 2.28305*y**2 + 1.07233*y**3 - 5.38434*y**4
         - 0.62251*y**5 + 5.30260*y**6 - 2.09002*y**7)
    
    return a + b / R_V

In [ ]:
# Pre-compute A_lambda / A_V at each filter wavelength
A_ratio = np.array([cardelli_extinction(w) for w in filter_wavelengths])

# Noise level: 5% of peak flux
noise_frac = 0.05

In [ ]:
def simulate_sed(theta):
    """
    Simulate noisy ugriz broadband photometry.
    
    Parameters
    ----------
    theta : torch.Tensor, shape (3,)
        theta[0] = T_eff          (effective temperature, K)
        theta[1] = log10(Omega)   (log10 of the solid angle)
        theta[2] = A_V            (V-band extinction, magnitudes)
    
    Returns
    -------
    data : torch.Tensor, shape (5,)
        Noisy flux values in each filter.
    """
    T     = theta[0].item()
    Omega = 10 ** theta[1].item()
    A_V   = theta[2].item()
    
    # TODO: Compute the Planck function B_lambda at each filter wavelength
    # TODO: Scale by Omega to get the intrinsic flux
    # TODO: Apply dust extinction: flux_reddened = flux * 10^(-0.4 * A_lambda)
    #       where A_lambda = A_V * A_ratio (pre-computed above)
    # TODO: Add Gaussian noise (std = noise_frac * max flux)
    # TODO: Return as a torch.float32 tensor
    
    return data

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Hint 1: Planck function at filter wavelengths</summary><div style="margin-top: 10px;">

The Planck function evaluated at the filter effective wavelengths is:

```python
exponent = np.clip(h_pl * c_l / (filter_wavelengths * kB * T), None, 500)
B_lambda = (2 * h_pl * c_l**2 / filter_wavelengths**5) / (np.exp(exponent) - 1)
```

The `np.clip` guards against overflow for very low temperatures.

</div></details>

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Hint 2: Applying extinction</summary><div style="margin-top: 10px;">

Extinction in magnitudes converts to a flux ratio via:

$$F_\text{reddened} = F_\text{intrinsic} \times 10^{-0.4 \, A_\lambda}$$

where $A_\lambda = A_V \times (A_\lambda / A_V)$ and the ratio $A_\lambda / A_V$ is pre-computed in `A_ratio`.

```python
A_lambda = A_V * A_ratio
flux_reddened = flux * 10**(-0.4 * A_lambda)
```

</div></details>

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution</summary><div style="margin-top: 10px;">

```python
def simulate_sed(theta):
    """
    Simulate noisy ugriz broadband photometry.
    
    Parameters
    ----------
    theta : torch.Tensor, shape (3,)
        theta[0] = T_eff          (effective temperature, K)
        theta[1] = log10(Omega)   (log10 of the solid angle)
        theta[2] = A_V            (V-band extinction, magnitudes)
    
    Returns
    -------
    data : torch.Tensor, shape (5,)
        Noisy flux values in each filter.
    """
    T     = theta[0].item()
    Omega = 10 ** theta[1].item()
    A_V   = theta[2].item()
    
    # Planck function at filter wavelengths
    exponent = np.clip(h_pl * c_l / (filter_wavelengths * kB * T), None, 500)
    B_lambda = (2 * h_pl * c_l**2 / filter_wavelengths**5) / (np.exp(exponent) - 1)
    
    # Intrinsic flux
    flux = Omega * B_lambda
    
    # Apply dust extinction
    A_lambda = A_V * A_ratio
    flux_reddened = flux * 10**(-0.4 * A_lambda)
    
    # Add noise
    noise_std = noise_frac * np.max(flux_reddened)
    flux_noisy = flux_reddened + noise_std * np.random.randn(len(filter_wavelengths))
    
    return torch.tensor(flux_noisy, dtype=torch.float32)
```

</div></details>

In [ ]:
def simulate_sed(theta):
    """
    Simulate noisy ugriz broadband photometry.
    
    Parameters
    ----------
    theta : torch.Tensor, shape (3,)
        theta[0] = T_eff          (effective temperature, K)
        theta[1] = log10(Omega)   (log10 of the solid angle)
        theta[2] = A_V            (V-band extinction, magnitudes)
    
    Returns
    -------
    data : torch.Tensor, shape (5,)
        Noisy flux values in each filter.
    """
    T     = theta[0].item()
    Omega = 10 ** theta[1].item()
    A_V   = theta[2].item()
    
    # Planck function at filter wavelengths
    exponent = np.clip(h_pl * c_l / (filter_wavelengths * kB * T), None, 500)
    B_lambda = (2 * h_pl * c_l**2 / filter_wavelengths**5) / (np.exp(exponent) - 1)
    
    # Intrinsic flux
    flux = Omega * B_lambda
    
    # Apply dust extinction
    A_lambda = A_V * A_ratio
    flux_reddened = flux * 10**(-0.4 * A_lambda)
    
    # Add noise
    noise_std = noise_frac * np.max(flux_reddened)
    flux_noisy = flux_reddened + noise_std * np.random.randn(len(filter_wavelengths))
    
    return torch.tensor(flux_noisy, dtype=torch.float32)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>Step 2: Generate a fake observation</h2>

Create a ground-truth observation with $T_\text{eff} = 6500$ K (an F-type star), $\log_{10}\Omega = -20$, and $A_V = 0.8$ mag (moderate extinction). Plot the noisy photometry alongside the noiseless reddened and unreddened SEDs to see the effect of dust.

</div>

In [ ]:
np.random.seed(42)

# Ground truth
T_true        = 6500.0
logOmega_true = -20.0
AV_true       = 0.8

theta_true_A = torch.tensor([T_true, logOmega_true, AV_true])
x_obs_A = simulate_sed(theta_true_A)

# TODO: Compute the noiseless reddened and unreddened SEDs for plotting
# TODO: Plot all three (unreddened as dotted, reddened as dashed, noisy as scatter)
# Hint: plot against filter_wavelengths * 1e9 for nm

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution</summary><div style="margin-top: 10px;">

```python
np.random.seed(42)

T_true        = 6500.0
logOmega_true = -20.0
AV_true       = 0.8

theta_true_A = torch.tensor([T_true, logOmega_true, AV_true])
x_obs_A = simulate_sed(theta_true_A)

# Noiseless SEDs
Omega_true = 10**logOmega_true
exponent = np.clip(h_pl * c_l / (filter_wavelengths * kB * T_true), None, 500)
B_true = (2 * h_pl * c_l**2 / filter_wavelengths**5) / (np.exp(exponent) - 1)
flux_intrinsic = Omega_true * B_true
flux_reddened  = flux_intrinsic * 10**(-0.4 * AV_true * A_ratio)

wl_nm = filter_wavelengths * 1e9

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(wl_nm, flux_intrinsic, 'k:', lw=1, alpha=0.4, marker='s', ms=5, label='Intrinsic (no dust)')
ax.plot(wl_nm, flux_reddened,  'k--', lw=1, alpha=0.6, marker='s', ms=5, label=f'Reddened ($A_V = {AV_true}$)')
ax.scatter(wl_nm, x_obs_A.numpy(), c='C0', s=50, zorder=5, label='Noisy observation')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Flux')
ax.set_title(rf'Observed SED ($T_{{\rm eff}} = {T_true:.0f}$ K, $\log_{{10}}\Omega = {logOmega_true}$, $A_V = {AV_true}$)')
ax.legend(fontsize=9)

# Label each filter
for i, name in enumerate(filter_names):
    ax.annotate(name, (wl_nm[i], x_obs_A[i].item()), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=9, color='C0')

plt.tight_layout()
plt.show()
```

</div></details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>Step 3: Define prior, simulate, train, and sample</h2>

Now set up the full SBI pipeline. Use:
<ul>
<li>Prior: $T_\text{eff} \in [3000, 12000]$ K, $\log_{10}\Omega \in [-22, -18]$, $A_V \in [0, 3]$ mag</li>
<li>10,000 training simulations</li>
<li>NPE with default settings</li>
</ul>

Then sample the posterior and make a corner plot with the true values overlaid.

</div>

In [ ]:
# TODO: Define the prior (3 parameters now)
prior_A = # ???

# TODO: Generate training simulations
n_sims = 10_000
thetas_A = # ???
xs_A     = # ???

# TODO: Train NPE
inference_A = # ???
# ???
density_estimator_A = # ???

# TODO: Build posterior and sample
posterior_A = # ???
samples_A   = # ???

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Hint 1: Prior</summary><div style="margin-top: 10px;">

Three parameters now:

```python
prior_A = BoxUniform(
    low=torch.tensor([3000.0, -22.0, 0.0]),
    high=torch.tensor([12000.0, -18.0, 3.0])
)
```

</div></details>

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution</summary><div style="margin-top: 10px;">

```python
# 1. Prior
prior_A = BoxUniform(
    low=torch.tensor([3000.0, -22.0, 0.0]),
    high=torch.tensor([12000.0, -18.0, 3.0])
)

# 2. Simulate
n_sims = 10_000
thetas_A = prior_A.sample((n_sims,))
xs_A = torch.stack([simulate_sed(t) for t in tqdm(thetas_A, desc='Simulating')])

# 3. Train
inference_A = NPE(prior=prior_A)
inference_A.append_simulations(thetas_A, xs_A)
density_estimator_A = inference_A.train(show_train_summary=True)

# 4. Sample
posterior_A = inference_A.build_posterior(density_estimator_A)
samples_A = posterior_A.sample((10_000,), x=x_obs_A)

# 5. Corner plot
fig = corner.corner(
    samples_A.numpy(),
    labels=[r'$T_{\rm eff}$ (K)', r'$\log_{10}\Omega$', r'$A_V$ (mag)'],
    truths=[T_true, logOmega_true, AV_true],
    truth_color='C3',
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt='.2f'
)
plt.suptitle('Track A: SED Fitting NPE Posterior', y=1.02, fontsize=14)
plt.show()
```

</div></details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>Step 4: Posterior predictive check</h2>

Draw 200 parameter vectors from the posterior, compute the noiseless reddened SED for each, and overlay them on the observed data. Do the posterior SEDs envelope the observation?

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

n_pp = 200
pp_samples_A = # ???

for i in range(n_pp):
    # TODO: compute noiseless reddened flux for this sample
    # TODO: plot with low alpha
    pass

wl_nm = filter_wavelengths * 1e9
ax.scatter(wl_nm, x_obs_A.numpy(), c='C3', s=50, zorder=5,
           edgecolors='k', linewidths=0.5, label='Observed data')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Flux')
ax.set_title('Posterior predictive check')
ax.legend()
plt.tight_layout()
plt.show()


<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution</summary><div style="margin-top: 10px;">

```python
fig, ax = plt.subplots(figsize=(8, 4))

n_pp = 200
pp_samples_A = posterior_A.sample((n_pp,), x=x_obs_A).numpy()

for i in range(n_pp):
    T_i     = pp_samples_A[i, 0]
    Omega_i = 10 ** pp_samples_A[i, 1]
    AV_i    = pp_samples_A[i, 2]
    
    exponent = np.clip(h_pl * c_l / (filter_wavelengths * kB * T_i), None, 500)
    B_i = (2 * h_pl * c_l**2 / filter_wavelengths**5) / (np.exp(exponent) - 1)
    flux_i = Omega_i * B_i * 10**(-0.4 * AV_i * A_ratio)
    ax.plot(wl_nm, flux_i, 'o-', color='C0', alpha=0.03, ms=3, lw=0.8)

ax.plot(wl_nm, flux_reddened, 'k--', lw=1.5, alpha=0.5, marker='s', ms=5, label='True reddened SED')
ax.scatter(wl_nm, x_obs_A.numpy(), c='C3', s=50, zorder=5,
           edgecolors='k', linewidths=0.5, label='Observed data')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Flux')
ax.set_title('Posterior predictive check: 200 SEDs from the posterior')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
```

</div></details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<hr>

<h1>Track B: Exoplanet Transit</h1>

<h2>The problem</h2>

You observe a stellar light curve during a planetary transit. The planet crosses the stellar disc, blocking a fraction of the starlight and producing a characteristic dip. We use a simplified (but physical) transit model with two parameters:

<ul>
<li>$R_p/R_\star$: the planet-to-star radius ratio. Controls the depth of the transit ($\delta \approx (R_p/R_\star)^2$).</li>
<li>$b$: the impact parameter (0 = central transit, 1 = grazing). Controls the transit duration and shape. Higher $b$ means shorter, V-shaped transits.</li>
</ul>

We fix the orbital period, semi-major axis, and stellar radius (i.e. we assume these are known from other data). The simulator generates a light curve at $n_t = 100$ equally-spaced times during a transit window, with Gaussian noise.

The interesting degeneracy: a larger planet on a more grazing orbit can produce a similar transit depth to a smaller planet on a central orbit, but the shape differs subtly. The posterior should capture this.

<h2>Step 1: Write the simulator</h2>

The transit model computes the overlap area between the planetary and stellar discs as a function of time. The key geometric quantity is the projected separation $z(t) = \sqrt{x(t)^2 + b^2}$, where $x(t)$ is the along-track position.

</div>

In [ ]:
n_t = 100 # number of time samples in the transit window

# Fixed orbital/stellar parameters
P       = 3.0       # orbital period (days)
a_Rs    = 8.0       # semi-major axis in units of stellar radii
noise_lc = 1.5e-3     # photometric noise (fractional)

# Time grid: centred on mid-transit, spanning a generous window
t_grid = np.linspace(-0.15, 0.15, n_t)  # days relative to mid-transit

In [ ]:
def uniform_transit_model(t, rp_rs, b, P=P, a_Rs=a_Rs):
    """
    Compute a uniform-source transit light curve using the
    analytic overlap formula for two circles.
    
    Parameters
    ----------
    t     : array, times in days relative to mid-transit
    rp_rs : float, planet-to-star radius ratio (R_p / R_star)
    b     : float, impact parameter (0 to 1)
    
    Returns
    -------
    flux : array, normalised flux (1 = no transit)
    """
    # Projected separation in units of R_star
    # x(t) = a/R_star * sin(2*pi*t/P),  z = sqrt(x^2 + b^2)
    x = a_Rs * np.sin(2 * np.pi * t / P)
    z = np.sqrt(x**2 + b**2)
    
    p = rp_rs  # shorthand
    
    flux = np.ones_like(z)
    for i in range(len(z)):
        zi = z[i]
        if zi >= 1 + p:
            # No overlap
            flux[i] = 1.0
        elif zi <= 1 - p:
            # Planet fully on disc
            flux[i] = 1.0 - p**2
        elif zi <= p - 1:
            # Star fully inside planet (shouldn't happen for p < 1, but just in case)
            flux[i] = 0.0
        else:
            # Partial overlap: area of intersection of two circles
            k0 = np.arccos((zi**2 + p**2 - 1) / (2 * zi * p))
            k1 = np.arccos((zi**2 + 1 - p**2) / (2 * zi))
            area = p**2 * k0 + k1 - 0.5 * np.sqrt(max(0, 4 * zi**2 - (1 + zi**2 - p**2)**2))
            flux[i] = 1.0 - area / np.pi
    
    return flux

In [ ]:
def simulate_transit(theta):
    """
    Simulate a noisy transit light curve.
    
    Parameters
    ----------
    theta : torch.Tensor, shape (2,)
        theta[0] = R_p / R_star   (radius ratio)
        theta[1] = b              (impact parameter)
    
    Returns
    -------
    data : torch.Tensor, shape (n_t,)
        Noisy normalised flux values.
    """
    rp_rs = theta[0].item()
    b     = theta[1].item()
    
    # TODO: Compute the transit model
    # TODO: Add Gaussian noise with std = noise_lc
    # TODO: Return as torch.float32 tensor
    
    return data

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Hint 1</summary><div style="margin-top: 10px;">

The transit model function is already written for you (`uniform_transit_model`). You just need to call it and add noise:

```python
flux_model = uniform_transit_model(t_grid, rp_rs, b)
```

</div></details>

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution</summary><div style="margin-top: 10px;">

```python
def simulate_transit(theta):
    """
    Simulate a noisy transit light curve.
    
    Parameters
    ----------
    theta : torch.Tensor, shape (2,)
        theta[0] = R_p / R_star   (radius ratio)
        theta[1] = b              (impact parameter)
    
    Returns
    -------
    data : torch.Tensor, shape (n_t,)
        Noisy normalised flux values.
    """
    rp_rs = theta[0].item()
    b     = theta[1].item()
    
    flux_model = uniform_transit_model(t_grid, rp_rs, b)
    flux_noisy = flux_model + noise_lc * np.random.randn(n_t)
    
    return torch.tensor(flux_noisy, dtype=torch.float32)
```

</div></details>

In [ ]:
def simulate_transit(theta):
    """
    Simulate a noisy transit light curve.
    
    Parameters
    ----------
    theta : torch.Tensor, shape (2,)
        theta[0] = R_p / R_star   (radius ratio)
        theta[1] = b              (impact parameter)
    
    Returns
    -------
    data : torch.Tensor, shape (n_t,)
        Noisy normalised flux values.
    """
    rp_rs = theta[0].item()
    b     = theta[1].item()
    
    flux_model = uniform_transit_model(t_grid, rp_rs, b)
    flux_noisy = flux_model + noise_lc * np.random.randn(n_t)
    
    return torch.tensor(flux_noisy, dtype=torch.float32)

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>Step 2: Generate a fake observation</h2>

Create a ground-truth observation with $R_p/R_\star = 0.1$ (a hot Jupiter) and $b = 0.4$ (a near-central transit). Plot the noisy light curve alongside the noiseless model.

</div>

In [ ]:
np.random.seed(42)

# Ground truth
rp_rs_true = 0.1
b_true     = 0.4

theta_true_B = torch.tensor([rp_rs_true, b_true])
x_obs_B = simulate_transit(theta_true_B)

# TODO: Also compute the noiseless light curve for plotting
# TODO: Plot both (noisy as scatter, noiseless as dashed line)
# Hint: convert t_grid to hours for the x-axis

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution</summary><div style="margin-top: 10px;">

```python
np.random.seed(42)

rp_rs_true = 0.1
b_true     = 0.4

theta_true_B = torch.tensor([rp_rs_true, b_true])
x_obs_B = simulate_transit(theta_true_B)

flux_true_B = uniform_transit_model(t_grid, rp_rs_true, b_true)

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(t_grid * 24, x_obs_B.numpy(), c='C0', s=20, zorder=5, label='Noisy observation')
ax.plot(t_grid * 24, flux_true_B, 'k--', lw=1, alpha=0.5, label='True light curve')
ax.set_xlabel('Time from mid-transit (hours)')
ax.set_ylabel('Normalised flux')
ax.set_title(rf'Observed transit ($R_p/R_\star = {rp_rs_true}$, $b = {b_true}$)')
ax.legend()
plt.tight_layout()
plt.show()
```

</div></details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>Step 3: Define prior, simulate, train, and sample</h2>

Set up the full pipeline. Use:
<ul>
<li>Prior: $R_p/R_\star \in [0.01, 0.25]$, $b \in [0.0, 1.0]$</li>
<li>10,000 training simulations</li>
<li>NPE with default settings</li>
</ul>

Then sample the posterior and make a corner plot.

</div>

In [ ]:
# TODO: Define the prior
prior_B = # ???

# TODO: Generate training simulations
n_sims = 10_000
thetas_B = # ???
xs_B     = # ???

# TODO: Train NPE
inference_B = # ???
# ???
density_estimator_B = # ???

# TODO: Build posterior and sample
posterior_B = # ???
samples_B   = # ???

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Hint 1: Prior bounds</summary><div style="margin-top: 10px;">

```python
prior_B = BoxUniform(
    low=torch.tensor([0.01, 0.0]),
    high=torch.tensor([0.25, 1.0])
)
```

Note: $b > 1$ means the planet misses the star entirely, so we cap at 1.0. In reality you might want to allow slightly grazing transits with $b$ up to $1 + R_p/R_\star$, but for simplicity we truncate at 1.

</div></details>

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution</summary><div style="margin-top: 10px;">

```python
# 1. Prior
prior_B = BoxUniform(
    low=torch.tensor([0.01, 0.0]),
    high=torch.tensor([0.25, 1.0])
)

# 2. Simulate
n_sims = 10_000
thetas_B = prior_B.sample((n_sims,))
xs_B = torch.stack([simulate_transit(t) for t in tqdm(thetas_B, desc='Simulating')])

# 3. Train
inference_B = NPE(prior=prior_B)
inference_B.append_simulations(thetas_B, xs_B)
density_estimator_B = inference_B.train(show_train_summary=True)

# 4. Sample
posterior_B = inference_B.build_posterior(density_estimator_B)
samples_B = posterior_B.sample((10_000,), x=x_obs_B)

# 5. Corner plot
fig = corner.corner(
    samples_B.numpy(),
    labels=[r'$R_p / R_\star$', r'$b$'],
    truths=[rp_rs_true, b_true],
    truth_color='C3',
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_fmt='.3f',
)
plt.suptitle('Track B: Transit NPE Posterior', y=1.02, fontsize=14)
plt.show()
```

</div></details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>Step 4: Posterior predictive check</h2>

Draw 200 parameter vectors from the posterior, compute the noiseless light curve for each, and overlay them on the observed data.

</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

n_pp = 200
pp_samples_B = # ???

for i in range(n_pp):
    # TODO: compute noiseless light curve for this sample
    # TODO: plot with low alpha
    pass

ax.scatter(t_grid * 24, x_obs_B.numpy(), c='C3', s=20, zorder=5,
           edgecolors='k', linewidths=0.5, label='Observed data')
ax.set_xlabel('Time from mid-transit (hours)')
ax.set_ylabel('Normalised flux')
ax.set_title('Posterior predictive check')
ax.legend()
plt.tight_layout()
plt.show()

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution</summary><div style="margin-top: 10px;">

```python
fig, ax = plt.subplots(figsize=(9, 4))

n_pp = 200
pp_samples_B = posterior_B.sample((n_pp,), x=x_obs_B).numpy()

for i in range(n_pp):
    rp_rs_i = pp_samples_B[i, 0]
    b_i     = pp_samples_B[i, 1]
    flux_i  = uniform_transit_model(t_grid, rp_rs_i, b_i)
    ax.plot(t_grid * 24, flux_i, color='C0', alpha=0.03, lw=1)

ax.scatter(t_grid * 24, x_obs_B.numpy(), c='C3', s=20, zorder=5,
           edgecolors='k', linewidths=0.5, label='Observed data')
ax.plot(t_grid * 24, flux_true_B, 'k--', lw=1.5, alpha=0.5, label='True light curve')
ax.set_xlabel('Time from mid-transit (hours)')
ax.set_ylabel('Normalised flux')
ax.set_title('Posterior predictive check: 200 light curves from the posterior')
ax.legend()
plt.tight_layout()
plt.show()
```

</div></details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§3. Posterior Predictive Checks</h1>

You already performed a posterior predictive check (PPC) in the exercises above. This is a visual sanity check. If the posterior produces data that doesn't resemble what you observed, something is wrong.

PPCs are a necessary check in the following sense: if the posterior is completely wrong (e.g. you accidentally used the prior as the posterior), the simulated data will not look like your observation. PPCs will catch that.

However, PPCs are not a sufficient check for calibration. A posterior can pass a PPC while still being overconfident or underconfident. For example, a posterior that is too narrow (overconfident) will still produce data that looks roughly correct, because the mean is in the right place. The coverage will be wrong, but you won't see that from a single PPC.

This brings us to the question: is the posterior actually calibrated? When the NPE says "90% credible interval", does it really contain the true parameter 90% of the time?

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§4. Simulation-Based Calibration</h1>

<h2>§4.1 The idea</h2>

Simulation-based calibration (SBC; <a href="https://arxiv.org/abs/1804.06788">Talts et al. 2018</a>) is the standard diagnostic for checking whether a posterior estimator is well-calibrated. The core idea follows directly from the structure of Bayesian inference.

Suppose you have a well-calibrated posterior $p(\theta \mid \mathbf{x})$. Consider the following generative process:

<ol>
<li>Draw $\theta^\star \sim \pi(\theta)$ from the prior.</li>
<li>Simulate $\mathbf{x}^\star \sim p(\mathbf{x} \mid \theta^\star)$.</li>
<li>Obtain posterior samples $\{\theta_1, \ldots, \theta_L\} \sim p(\theta \mid \mathbf{x}^\star)$.</li>
<li>Compute the <b>rank</b> of $\theta^\star$ among the posterior samples. For each parameter dimension separately, count how many of the $L$ posterior samples have a value less than $\theta^\star$. For example, if $\theta^\star = 25$ m/s and 600 out of 1000 posterior samples are below 25 m/s, then the rank is 600. If $\theta^\star$ happens to sit right in the middle of the posterior, the rank will be around $L/2$. If the posterior is too narrow and misses $\theta^\star$ entirely on the low side, the rank will be close to $L$ (almost all samples are below the truth). If it misses on the high side, the rank will be close to 0.</li></ol>

If the posterior is perfectly calibrated, the rank of $\theta^\star$ should be uniformly distributed between 0 and $L$. This is because, under the true generative model, $\theta^\star$ is just another draw from the posterior, so it is equally likely to fall anywhere in the distribution.

The diagnostic is: repeat steps 1-4 many times (say, 200-1000), collect the ranks, and check if they are uniformly distributed. If they are, the posterior is well-calibrated. If not, the pattern of non-uniformity tells you what is wrong:

<ul>
<li>U-shaped histogram (too many ranks near 0 and $L$): the posterior is overconfident (too narrow). The true parameter falls outside the posterior more often than it should.</li>
<li>Inverse-U / hump-shaped histogram: the posterior is underconfident (too wide). The posterior is wider than it needs to be.</li>
<li>Skewed histogram: the posterior has a systematic bias. It is shifted relative to the truth.</li>
<li>Flat histogram: the posterior is well-calibrated.</li>
</ul>

Importantly, SBC checks a necessary condition. If SBC fails, the posterior is definitely wrong. But if SBC passes, that doesn't guarantee correctness. A trivial example: using the prior as the posterior gives perfectly uniform ranks (the prior is, by definition, "calibrated" in this sense), but the inference would be useless. This is why you need both SBC <i>and</i> PPCs. Together they cover the two failure modes: wrong shape (caught by SBC) and wrong location (caught by PPCs).

<h2>§4.2 SBC with <code>sbi</code></h2>

The <code>sbi</code> package provides <code>run_sbc</code> and <code>sbc_rank_plot</code> for this. Because NPE is amortised, SBC is cheap: each of the 200 test cases just needs a forward pass, not a new MCMC chain.

We will run SBC on the projectile example from §1, since we already have the trained posterior.

</div>

In [ ]:
# Generate test set: new (theta*, x*) pairs from the prior + simulator
num_sbc_samples = 1000
sbc_thetas = prior.sample((num_sbc_samples,))
sbc_xs = torch.stack([simulate_projectile(t) for t in tqdm(sbc_thetas, desc='SBC simulations')])

# Run SBC: for each test case, draw 1000 posterior samples and compute the rank
num_posterior_samples = 1_000
ranks, dap_samples = run_sbc(
    sbc_thetas, sbc_xs, posterior,
    num_posterior_samples=num_posterior_samples,
    use_batched_sampling=False,
)

# Plot the rank histograms
fig, axes = sbc_rank_plot(
    ranks, num_posterior_samples,
    num_bins=20,
    parameter_labels=[r'$v_0$', r'$\alpha$'],
    plot_type = 'hist',
    figsize=(8, 3),
)
fig.suptitle('SBC rank histograms — projectile NPE', y=1.05, fontsize=13)
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§4.3 Interpreting the result</h2>

If the histograms are approximately flat (within the expected statistical fluctuations, shown by the shaded confidence band), the posterior is well-calibrated.

If you see deviations, ask:

<ul>
<li>Is it U-shaped? Your posterior is overconfident. You may need more training simulations, a more expressive network, or to check your noise model.</li>
<li>Is it humped? Your posterior is underconfident. This is less common with NPE but can happen if the network hasn't converged.</li>
<li>Is it skewed? There is a systematic bias. Check whether the simulator has edge cases (e.g. the projectile guard clause for $t_\text{flight} \leq 0$) that might bias the training set.</li>
</ul>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§4.4 Exercise: SBC on your §2 exercise</h2>

Run SBC on whichever Track A or Track B problem you completed in §2. Use 200 test samples and 1000 posterior samples per test case.

<ol>
<li>Generate test pairs $(\theta^\star, \mathbf{x}^\star)$ from the prior and simulator.</li>
<li>Run <code>run_sbc</code>.</li>
<li>Plot the rank histograms.</li>
<li>Interpret: is the posterior well-calibrated? If not, what does the shape tell you?</li>
</ol>

</div>

In [ ]:
# TODO: Generate test pairs from the prior and simulator you used in §2
num_sbc_samples = 200
sbc_thetas_ex = # ???
sbc_xs_ex     = # ???

# TODO: Run SBC
ranks_ex, dap_ex = # ???

# TODO: Plot rank histograms

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution (Track A: SED fitting)</summary><div style="margin-top: 10px;">

```python
from sbi.diagnostics import run_sbc
from sbi.analysis.plot import sbc_rank_plot

num_sbc_samples = 200
sbc_thetas_A = prior_A.sample((num_sbc_samples,))
sbc_xs_A = torch.stack([simulate_sed(t) for t in tqdm(sbc_thetas_A, desc='SBC simulations')])

num_posterior_samples = 1_000
ranks_A, dap_A = run_sbc(
    sbc_thetas_A, sbc_xs_A, posterior_A,
    num_posterior_samples=num_posterior_samples,
    use_batched_sampling=False,
)

fig, axes = sbc_rank_plot(
    ranks_A, num_posterior_samples,
    num_bins=20,
    parameter_labels=[r'$T_{\rm eff}$', r'$\log_{10}\Omega$', r'$A_V$'],
    figsize=(10, 3),
)
fig.suptitle('SBC rank histograms — SED fitting NPE', y=1.05, fontsize=13)
plt.tight_layout()
plt.show()
```

</div></details>

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Full solution (Track B: Transit)</summary><div style="margin-top: 10px;">

```python
from sbi.diagnostics import run_sbc
from sbi.analysis.plot import sbc_rank_plot

num_sbc_samples = 200
sbc_thetas_B = prior_B.sample((num_sbc_samples,))
sbc_xs_B = torch.stack([simulate_transit(t) for t in tqdm(sbc_thetas_B, desc='SBC simulations')])

num_posterior_samples = 1_000
ranks_B, dap_B = run_sbc(
    sbc_thetas_B, sbc_xs_B, posterior_B,
    num_posterior_samples=num_posterior_samples,
    use_batched_sampling=False,
)

fig, axes = sbc_rank_plot(
    ranks_B, num_posterior_samples,
    num_bins=20,
    parameter_labels=[r'$R_p / R_\star$', r'$b$'],
    figsize=(8, 3),
)
fig.suptitle('SBC rank histograms — Transit NPE', y=1.05, fontsize=13)
plt.tight_layout()
plt.show()
```

</div></details>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§4.5 The data-averaged posterior (DAP)</h2>

SBC gives us a second useful diagnostic for free: the data-averaged posterior (DAP). For each of the SBC test cases, we saved one random posterior sample. The collection of these single samples, drawn across many different observations, should recover the prior distribution (this follows from the same self-consistency argument as the rank uniformity).

If the DAP matches the prior, the posterior is consistent on average. If it doesn't, it reveals systematic biases.

</div>

In [ ]:
# Plot DAP vs prior for the projectile example
param_labels = [r'$v_0$ (m/s)', r'$\alpha$ (rad)']
n_params = sbc_thetas.shape[1]

fig, axes = plt.subplots(1, n_params, figsize=(5 * n_params, 4))
if n_params == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # DAP samples
    ax.hist(dap_samples[:, i].numpy(), bins=30, density=True,
            alpha=0.6, color='C0', label='DAP')
    # Prior samples
    prior_draws = prior.sample((5000,))[:, i].numpy()
    ax.hist(prior_draws, bins=30, density=True,
            alpha=0.3, color='k', label='Prior')
    ax.set_xlabel(param_labels[i])
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

fig.suptitle('Data-averaged posterior vs prior', fontsize=13)
plt.tight_layout()
plt.show()


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§4.6 Building intuition</h2>

<h3>Why should the ranks be uniform?</h3>

Think about what we're doing in SBC. We draw $\theta^\star$ from the prior, simulate data from it, and then ask: where does $\theta^\star$ sit in the posterior we get back?

If the posterior is doing its job correctly, then $\theta^\star$ is just another sample from the posterior. This follows from the definition of the Bayesian generative model: $\theta^\star$ was drawn from the prior, the data was generated from $\theta^\star$, and the posterior is the correct update of the prior given that data. So $\theta^\star$ and the posterior samples are all draws from the same distribution.

Now, if you take a random draw from a distribution and ask "what fraction of the distribution is below me?", the answer is uniformly distributed between 0 and 1. This is the probability integral transform. The rank is just the discrete version of this: out of $L$ posterior samples, how many are below $\theta^\star$? If $\theta^\star$ is a genuine draw from the posterior, this count is uniform over $\{0, 1, \ldots, L\}$.

<h3>The data-averaged posterior</h3>

The DAP is a complementary check that falls out of the SBC procedure for free. During SBC, for each test case we draw many posterior samples. If we keep just one random sample per test case, we get a collection of single draws from many different posteriors, each conditioned on different data.

This collection should look like the prior. Why?

Each posterior sample $\theta_i$ is drawn from $p(\theta \mid \mathbf{x}_i^\star)$, and each $\mathbf{x}_i^\star$ was generated from $\theta_i^\star \sim \pi(\theta)$. Marginalising over the data:

$$p(\theta) = \int p(\theta \mid \mathbf{x}) \, p(\mathbf{x}) \, d\mathbf{x}$$

This is just the prior. So if you average the posterior over all possible datasets (drawn from the prior predictive), you recover the prior. The DAP is the empirical version of this integral.

If the DAP doesn't match the prior, the posterior is systematically biased. For example, if the DAP is narrower than the prior, the posteriors are collectively overconfident. If it's shifted, there's a systematic bias.

<h3>Summary table</h3>

<table style="width: 100%; border-collapse: collapse; margin: 20px 0;">
<tr>
<th style="padding: 10px; text-align: left; border: 1px solid rgba(0,64,133,0.3);">Diagnostic</th>
<th style="padding: 10px; text-align: left; border: 1px solid rgba(0,64,133,0.3);">What it checks</th>
<th style="padding: 10px; text-align: left; border: 1px solid rgba(0,64,133,0.3);">Necessary or sufficient?</th>
</tr>
<tr>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);"><b>Posterior predictive check</b></td>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);">Does the posterior produce data that looks like the observation?</td>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);">Necessary (if the posterior is completely wrong, the simulated data won't match the observation), but not sufficient (an overconfident posterior can still pass if the mean is in the right place).</td>
</tr>
<tr>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);"><b>SBC rank uniformity</b></td>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);">Are the credible intervals correctly calibrated?</td>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);">Necessary (if it fails, the posterior is wrong), but not sufficient (the prior itself passes SBC).</td>
</tr>
<tr>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);"><b>Data-averaged posterior</b></td>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);">Does the marginal distribution of posterior samples recover the prior?</td>
<td style="padding: 10px; border: 1px solid rgba(0,64,133,0.3);">Necessary (same logic as SBC).</td>
</tr>
</table>

The combination of PPCs + SBC covers the two main failure modes. Always run both before trusting an SBI posterior.

</div>
<h2>§4.7 When SBC fails: what to do</h2>

If SBC reveals that your posterior is miscalibrated, here are the most common fixes, roughly in order of how often they help:

<ol>
<li>More training simulations. This is the single most common fix. 10,000 sims is fine for 2-3 parameter toy problems, but real problems with higher-dimensional data or more complex posteriors may need 50,000-100,000+.</li>
<li>More expressive network. Try a neural spline flow (NSF) instead of the default MAF, or increase the number of transforms/hidden units.</li>
<li>Better prior. If the prior is much wider than the region where the posterior has support, most training simulations are wasted. Tighten the prior if you have domain knowledge, or use sequential NPE (SNPE) to focus simulations.</li>
<li>Check the simulator. Edge cases (division by zero, clipping, guard clauses) can create discontinuities in the data distribution that the flow struggles to learn.</li>
<li>Embedding network. If the data is high-dimensional, the default network may struggle without a learned compression step.</li>
</ol>

</div>